In [ ]:
from river import drift
import numpy as np

from cmab.scm.distribution.bernoulli import Bernoulli
from cmab.scm.mechanism import Mechanism
from cmab.scm.mechanism import Mechanism
from cmab.scm.scm import SCM
from cmab.environments import CausalBanditEnv, NSCausalBanditEnv
from cmab.environments.ns.scheduling.controlled_schedule import ControlledShiftSchedule

In [22]:
SEED=43

### Data streams

In [ ]:
V = ['X', 'Z', 'Y']
U = ['U_X', 'U_Z', 'U_Y']

P_X = Bernoulli(p=0.6)
P_Z = Bernoulli(p=0.7)
P_Y = Bernoulli(p=0.9)

mechanism_X = Mechanism(v_parents=[], u_parents=['U_X'], f=lambda _, u: u['U_X'])
mechanism_Z = Mechanism(v_parents=[], u_parents=['U_Z'], f=lambda _, u: u['U_Z'])
mechanism_Y = Mechanism(v_parents=['X', 'Z'], u_parents=['U_Y'], f=lambda v, u: v['X'] ^ v['Z'] ^  u['U_Y'])

scm = SCM(
    U=U,
    V=V,
    P_u_marginals={
        'U_X': P_X,
        'U_Z': P_Z,
        'U_Y': P_Y
    },
    F={
        'X': mechanism_X,
        'Z': mechanism_Z,
        'Y': mechanism_Y
    },
    seed=SEED
)

schedule = ControlledShiftSchedule(exogenous=['U_X', 'U_X', 'U_X'], new_params=[0.75, 0.9, 0.8], every=500)
env = NSCausalBanditEnv(scm=scm, reward_node='Y', seed=SEED, atomic=True, schedule=schedule, include_empty=True)
print(f"Number of actions: {len(env.action_space)}")
print(f"Action space: {env.action_space}")

reward_stream = []
Z_stream = []
X_stream = []
T = 2000

for i in range(T):
    action = env.action_space[0]
    _, obs, _, _, _ = env.step(action)
    reward_stream.append(obs['Y'])
    Z_stream.append(obs['Z'])
    X_stream.append(obs['X'])

Number of actions: 5
Action space: [frozenset(), frozenset({('X', 0)}), frozenset({('X', 1)}), frozenset({('Z', 0)}), frozenset({('Z', 1)})]


In [24]:
# print expected values of each stream at each segment
print("Expected values of X stream at each segment:")
print(f"Segment 1 (0-499): {np.mean(X_stream[0:500])}")
print(f"Segment 2 (500-999): {np.mean(X_stream[500:1000])}")
print(f"Segment 3 (1000-1499): {np.mean(X_stream[1000:1500])}")
print(f"Segment 4 (1500-1999): {np.mean(X_stream[1500:2000])}")

Expected values of X stream at each segment:
Segment 1 (0-499): 0.56
Segment 2 (500-999): 0.672
Segment 3 (1000-1499): 0.736
Segment 4 (1500-1999): 0.79


In [25]:
# print expected values of each stream at each segment
print("Expected values of Z stream at each segment:")
print(f"Segment 1 (0-499): {np.mean(Z_stream[0:500])}")
print(f"Segment 2 (500-999): {np.mean(Z_stream[500:1000])}")
print(f"Segment 3 (1000-1499): {np.mean(Z_stream[1000:1500])}")
print(f"Segment 4 (1500-1999): {np.mean(Z_stream[1500:2000])}")

Expected values of Z stream at each segment:
Segment 1 (0-499): 0.692
Segment 2 (500-999): 0.714
Segment 3 (1000-1499): 0.688
Segment 4 (1500-1999): 0.67


### Evaluate PH Detector

In [53]:
delta = 0.001  # Sensitivity parameter for PH test
threshold = 25.0  # Threshold for PH test to signal a change
min_instances = 30

In [54]:
ph = drift.PageHinkley(delta=delta, threshold=threshold, min_instances=min_instances)

for i, val in enumerate(X_stream):
    ph.update(val)
    if ph.drift_detected:
        print(f"PH change detected at index {i}, input value: {val}")

PH change detected at index 809, input value: 1
PH change detected at index 1712, input value: 1


In [55]:
ph = drift.PageHinkley(delta=delta, threshold=threshold, min_instances=min_instances)

for i, val in enumerate(Z_stream):
    ph.update(val)
    if ph.drift_detected:
        print(f"PH change detected at index {i}, input value: {val}")

PH change detected at index 1593, input value: 0


In [56]:
ph = drift.PageHinkley(delta=delta, threshold=threshold, min_instances=min_instances)

for i, val in enumerate(reward_stream):
    ph.update(val)
    if ph.drift_detected:
        print(f"PH change detected at index {i}, input value: {val}")

PH change detected at index 1080, input value: 1
PH change detected at index 1646, input value: 0


### Evaluate BOCPD

In [ ]:
from cmab.algorithms.cpd.bodc import updateForecasterDistribution, updateForecasterDistribution_m, updateLaplacePrediction

In [ ]:
Horizon = T
gamma = 1 / Horizon  # hazard
alphas = np.array([1])
betas = np.array([1])
ForecasterDistribution = np.array([1])
ChangePointEstimation = np.array([])

In [ ]:
for i, val in enumerate(data_stream_X):
    EstimatedBestExpert = np.argmax(ForecasterDistribution) #Change-point estimation
    ChangePointEstimation = np.append(ChangePointEstimation,EstimatedBestExpert+1)
    ForecasterDistribution = updateForecasterDistribution(ForecasterDistribution, alphas, betas, val, gamma)
    (alphas, betas) = updateLaplacePrediction(alphas, betas, val) #Update the laplace predictor

print("Change Point Estimation:", len(ChangePointEstimation))

In [ ]:
    
from itertools import product

cpds = {}
cfg = list(product([0, 1], repeat=0))
for cfge in cfg:
                cpds[cfge] = "test"

print(cpds)

In [ ]:
observation = [0, 1, 0, 0, 1, 1, 0, 1, 0, 0]

In [ ]:
cfg = tuple(observation[parent] for parent in [])
print(cpds[cfg])